# Run the whole grid — 3 designs x 2 item sets

Runs all six calibrations back to back, `birt` -> `bmi` -> `flat` in each.
**Generated by `notebooks/make_design_notebooks.py` — edit the generator,
not this file.** All logic is in `mice_imp/run_grid.py`.

Order is CommonItem, Binterlaced, Standard, and `full` before `nobike`:
increasing difficulty, so a failure lands where there is the most signal to
diagnose it. `standard/nobike` is the thinnest cell in the grid and runs last.

The runner is **resumable** — a run whose three arm `.npz` files already
exist is skipped, so an interrupted weekend continues rather than restarting
— **isolated**, in that one run raising does not stop the other five, and
**logged**, with each run's output captured to
`results/<design>/<itemset>/run.log` and `results/grid_summary.csv`
rewritten after every run.

Everything this notebook writes goes under **`results/`**, and every
exported CSV row carries `variant = "main"`, so these estimates cannot be
pooled with the other grid's by accident.

## For a weekend run, prefer the terminal

A notebook kernel is the wrong container for something that has to survive
until Monday: a closed laptop, a suspended session or a dropped connection
takes the kernel and the run with it. From a terminal instead:

```
nohup python -m mice_imp.run_grid > results/grid.out 2>&1 &
tail -f results/grid.out
```

Same script, same resumability. Use the cells below to watch progress, or to
run a single cell of the grid interactively.

## Smoke first

Run the wiring check before committing a weekend to a full pass — it is
2 chains of 50/50 with a
20-epoch ADVI warm-start, against 4 x
3000/2000 for the real thing. It writes to the same
directories, so a real run afterwards needs `--force` (or delete the smoke
`.npz` files).

It is sized to answer *does the plumbing work*, not *what is the answer*: at
50 warmup draws the R-hats and the noise floor mean nothing.
Read only the three checks below off it. The floor on how fast it can go is
the B-MI imputation fit, which does not shrink with chain length; a re-run in
a directory that already has `imputation_model/` reuses it.

Override any of it with `--num-chains`, `--num-warmup`, `--num-samples`,
`--advi-iterations`.

Three things to read off it, each of which silently invalidates a run:

- `Initializing from ADVI solution` must appear for **all three** arms. If it
  appears for some and not others, the `flat`-vs-`birt` gap is measuring
  initialisation rather than Monte Carlo noise, and the noise floor is
  meaningless (CLAUDE.md fact 17).
- the **B-MI flat-fallback rate** must be low. Every fallback downgrades that
  cell to B-IRT, so a high rate means B-MI is not really running (fact 3).
- `flat` and `birt` must land on top of each other. They target the same
  posterior by construction; a systematic gap means the plumbing is wrong
  (fact 2).

In [ ]:
import subprocess, sys
from pathlib import Path

REPO = Path("/home/luis/CGE/mice_imp")

def run_grid(*flags):
    """Run mice_imp.run_grid as a subprocess, streaming its output."""
    cmd = [sys.executable, "-m", "mice_imp.run_grid", *flags]
    print(" ".join(cmd), flush=True)
    proc = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    return proc.wait()


# Wiring check only -- uncomment to run it. It writes .npz files, so the
# full pass below would then need --force.
# run_grid("--smoke", "--force")

## The full grid

4 chains, 3000 warmup, 2000 samples, six times over. Only start this once
the smoke output above looks right.

If you smoke-tested first, the smoke `.npz` files are sitting in the output
directories and will cause every run to be skipped as already complete —
hence `--force` here.

In [2]:
run_grid("--force")

/home/luis/.cache/pypoetry/virtualenvs/mice-imp-toZTZ-oV-py3.12/bin/python -m mice_imp.run_grid --force
grid: 6 run(s), smoke=False, force=True

CommonItem / full  (27 items, 4 chains, 3000/2000)
  log: /home/luis/CGE/mice_imp/results/commonitem/full/run.log

=== arm: birt ===
compat: patched bayesianquilts.util.training_loop for the batchless marginal-ADVI call (see mice_imp/compat.py).
    ADVI warm-start (100 iterations)
Marginal ADVI (Rao-Blackwellized abilities)
  Item vars: ['difficulties0', 'discriminations', 'ddifficulties']
  Surrogate params: 324
  Rank: 0 (mean-field)
--- Starting Training ---
Patience for early stopping: 3 epochs
LR decay factor on plateau: 0.5
Convergence will be checked every: 1 epoch(s)
Checkpoints will be saved to: None
Optimizing keys: ['mu\\identity\\normal\\loc', 'mu\\identity\\normal\\scale', 'difficulties0\\identity\\normal\\loc', 'difficulties0\\identity\\normal\\scale', 'discriminations\\softplus\\normal\\loc', 'discriminations\\softplus\\normal\

0

## Progress

Safe to run while the grid is still going: `results/grid_summary.csv` is
rewritten after every run, and each run's `run.log` is flushed as it goes.

In [ ]:
import pandas as pd

summary = pd.read_csv(REPO / "results" / "grid_summary.csv")
cols = [c for c in ["design", "variant", "itemset", "n_items", "status", "minutes",
                    "birt_max_rhat", "bmi_max_rhat", "flat_max_rhat",
                    "bmi_fallback_pct", "advi_init_count",
                    "theta_rmse_flat_vs_birt", "theta_rmse_bmi_vs_birt"]
        if c in summary.columns]
summary[cols]

## Re-run one cell

`--only` takes `design/itemset`, e.g. after fixing whatever made a single
run fail. Keep `--force` as needed on it — without that the re-run reads the
other variant's data and writes over the other variant's results:

```
run_grid("--only", "standard/nobike", "--force", "--force")
```

In [ ]:
# run_grid("--only", "standard/nobike", "--force", "--force")